# Sentiment Analysis using NLTK

In [5]:
'''
This script perfroms sentiment analysis using the NLTK Library in python. Please refer to the ReadMe file for relevant changes before running the script
'''

import pandas as pd
import re
import nltk
from textblob import TextBlob
import os

nltk_data_path = r'C:\nltk_data'
os.makedirs(nltk_data_path, exist_ok=True)
nltk.data.path.append(nltk_data_path)
nltk.download('stopwords', quiet=True, download_dir=nltk_data_path)
nltk.download('punkt', quiet=True, download_dir=nltk_data_path)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

try:
    test_sentence = "This is a test sentence."
    word_tokenize(test_sentence)
    stopwords.words('english')
    sent_tokenize(test_sentence)
    print("NLTK data loaded successfully.")
except LookupError as e:
    print(f"Error loading NLTK data: {e}")
    print("NLTK data paths:", nltk.data.path)
    raise

stop_words = set(stopwords.words('english'))

file_path = r'C:\Users\singh\Downloads\Reddit FIles\Comments\Cleaned_complete.xlsx'
df = pd.read_excel(file_path)

required_columns = ['body', 'crime_type', 'Location']
if not all(col in df.columns for col in required_columns):
    raise Exception("The XLSX file must contain 'body', 'crime_type', and 'Location' columns.")

criminal_keywords = {
    'crime': 5, 'robbed': 6, 'law enforcement': 3, 'murder': 9, 'police report': 2, 
    'hate crime': 8, 'rob': 6, 'bad neighborhood': 3, 'stolen': 4, 'theft': 4, 
    'assault': 7, 'assaulted': 7, 'riot': 6, 'harassment': 4, 'stabbing': 8, 
    'car theft': 4, 'arson': 7, 'robbery': 6, 'attempted murder': 8, 'serial killer': 10, 
    'kidnapping': 9, 'hit and run': 7, 'prostitution': 3, 'property damage': 4, 
    'homicide': 9, 'sexual harassment': 6, 'ransom': 8, 'sexual assault': 9, 
    'identity theft': 5, 'burglary': 5, 'vandalism': 3, 'home invasion': 7, 
    'extortion': 6, 'bribery': 4, 'self-defense': 2, 'smuggling': 6, 
    'organized crime': 7, 'human trafficking': 9, 'shoplifting': 2, 
    'money laundering': 6, 'death threat': 7, 'unsafe to walk': 3, 'trafficking': 8, 
    'breaking and entering': 5, 'drug dealing': 5, 'looting': 4, 'street racing': 3, 
    'illegal firearms': 8, 'public safety': 3, 'drug trafficking': 7, 
    'cyber attack': 6, 'terrorist attack': 10, 'abduction': 9, 'bomb threat': 9, 
    'illegal weapons': 8, 'illegal gambling': 4, 'police brutality': 5, 
    'hostage situation': 9, 'serial killer': 10, 'shootout': 8, 
    'public disturbance': 2, 'neighborhood safety': 2, 'threatening behavior': 5
}

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www.\S+', '', text)
    text = re.sub(r'@\w+|#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    text = ' '.join(word for word in tokens if word not in stop_words)
    return text

df['cleaned_content'] = df['body'].apply(preprocess_text)

def enhanced_analyze_sentiment_and_rating(text, crime_type):
    analysis = TextBlob(text)
    polarity = analysis.sentiment.polarity
    words = text.lower().split() + crime_type.lower().split()
    criminal_intensity = sum(criminal_keywords.get(word, 0) for word in words)
    severe_term_count = len(re.findall(r'\b(murder|kill|rob|assault|crime|violence|stabbing)\w*\b', text.lower() + ' ' + crime_type.lower()))
    criminal_intensity += severe_term_count * 3
    max_possible_intensity = sum(max(criminal_keywords.values()) for _ in range(len(words)))
    normalized_intensity = criminal_intensity / max_possible_intensity if max_possible_intensity > 0 else 0
    adjusted_polarity = polarity - (normalized_intensity * 1.2)
    base_score = int((1 - normalized_intensity) * 15)
    if re.search(r'\b(be careful|watch out|dangerous|avoid|avoiding)\b', text.lower()):
        base_score -= 4
    if re.search(r'\b(higher crime|high crime|crime is unavoidable)\b', text.lower()):
        base_score -= 3
    rating = max(0, min(base_score, 20))
    if adjusted_polarity > 0.1:
        sentiment = 'Positive'
    elif adjusted_polarity < -0.1:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
    return sentiment, adjusted_polarity, normalized_intensity, rating

df['sentiment'], df['adjusted_polarity'], df['criminal_intensity'], df['rating'] = zip(*df.apply(lambda row: enhanced_analyze_sentiment_and_rating(row['cleaned_content'], row['crime_type']), axis=1))
output_path = r'C:\Users\singh\Downloads\EAS508 Project\Sentiment Analysis\Complete_With_Sentiment_And_Intent_Rating.xlsx'
df[['Location', 'body', 'crime_type', 'cleaned_content', 'sentiment', 'adjusted_polarity', 'criminal_intensity', 'rating']].to_excel(output_path, index=False)
print(f"Sentiment analysis and rating assignment completed. Results saved to {output_path}.")
print(df[['Location', 'body', 'crime_type', 'cleaned_content', 'sentiment', 'adjusted_polarity', 'criminal_intensity', 'rating']].head())


NLTK data loaded successfully.
Sentiment analysis and rating assignment completed. Results saved to C:\Users\singh\Downloads\Reddit FIles\Comments\Final Sentiment Analysis Files\Complete_with_sentiment_and_intent_rating.xlsx.
  Location                                               body crime_type  \
0  Alabama  Helena thinks they are the least crime in all ...      crime   
1  Alabama                    Next up. Jerking off is murder.     murder   
2  Alabama                     Yelp is An extortion platform.  extortion   
3  Alabama  Seems like everyone here thinks that they know...     stolen   
4  Alabama  All the more reason to not commit capital murder.     murder   

                               cleaned_content sentiment  adjusted_polarity  \
0            helena thinks least crime alabama  Negative             -0.620   
1                          next jerking murder  Negative             -0.720   
2                      yelp extortion platform  Negative             -0.360   
3